In [19]:
import pandas as pd
import os
from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device: ", DEVICE)

Device:  cpu


In [ ]:
DATA_PATH = r"datasets"
df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
df.head()

Shape:  (3662, 2)
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0


In [ ]:
print("Shape: ", df.shape)
print(df['diagnosis'].value_counts().sort_index())

Shape:  (3662, 2)
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64


In [18]:
train_df, temp_df = train_test_split(df, test_size = 0.3, stratify = df['diagnosis'], random_state = 35)
val_df, test_df = train_test_split(temp_df, test_size = 0.5, stratify = temp_df['diagnosis'], random_state = 35)

In [20]:
class retinalDataset(Dataset):
    def __init__(self, df, images_dir, transform = None):
        self.df = df
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, f"{row['id_code']}.png")

        image = Image.open(path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = int(row['diagnosis'])
        return image, label